## Summary
This notebook merges locality and SA2 income tables, audits the merged data (missingness, duplicates, many-to-many joins), and identifies income outliers at the SA2 level using IQR and z‑score methods. It also bins SA2s into income quintiles and maps results back to localities for downstream use.

In [ ]:
# Consolidated imports and display options
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [ ]:
sa2_income = pd.read_csv("../data/income/sa2_income.csv")
locality_to_sa2 = pd.read_csv("../data/income/2024 Locality to 2021 SA2 Coding Index.csv")

# Align key names
sa2_income = sa2_income.rename(columns={
    'Statistical Areas Level 2 2021 code': 'SA2_CODE_2021',
    'Statistical Areas Level 2 2021 name': 'SA2_NAME_2021',
})

# Coerce keys to consistent dtypes and trim whitespace
for df in (sa2_income, locality_to_sa2):
    for col in ('SA2_CODE_2021', 'SA2_NAME_2021'):
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# Choose an income column from the 2024 DbR release (personal median total income)
income_candidates = [
    c for c in sa2_income.columns
    if c.startswith('Personal income: Median total income (excl. Government pensions and allowances)')
]
INCOME_COL = income_candidates[0] if income_candidates else None

cols_to_keep = ['SA2_CODE_2021', 'SA2_NAME_2021'] + ([INCOME_COL] if INCOME_COL else [])

merged = locality_to_sa2.merge(
    sa2_income[cols_to_keep],
    on=['SA2_CODE_2021', 'SA2_NAME_2021'],
    how='left'
)

print('Using income column:', INCOME_COL)
print(merged.shape)
merged.head(3)

In [ ]:
# Missing values overview
na_counts = merged.isna().sum().sort_values(ascending=False)
na_pct = (merged.isna().mean() * 100).round(2).sort_values(ascending=False)

na_summary = (
    pd.concat([na_counts.rename('missing_count'), na_pct.rename('missing_pct')], axis=1)
    .sort_values(['missing_count', 'missing_pct'], ascending=False)
)

print(merged.shape)
na_summary.head(30)


In [ ]:
# Duplicate analysis
# 1) Full-row duplicates
full_dupe_count = merged.duplicated().sum()
print(f"Full-row duplicates: {full_dupe_count}")

# 2) Key subset duplicates: choose a join key
# Guessing columns from merged file header
candidate_keys = [
    'POSTCODE',
    'LOCALITY_NAME',
    'STATE',
    'SA2_CODE_2021',
    'SA2_NAME_2021'
]
existing_keys = [c for c in candidate_keys if c in merged.columns]
print('Existing key columns:', existing_keys)

if existing_keys:
    key_dupes = merged.duplicated(subset=existing_keys, keep=False)
    dupe_groups = (
        merged.loc[key_dupes, existing_keys]
        .value_counts()
        .reset_index(name='rows_per_key')
        .sort_values('rows_per_key', ascending=False)
    )
    print('Rows with duplicate keys:', key_dupes.sum())
    dupe_groups.head(20)
else:
    print('No expected key columns found; adjust candidate_keys if needed.')


In [ ]:
# Relationship checks between locality and SA2
cols = merged.columns
has_locality = 'LOCALITY_NAME' in cols and 'POSTCODE' in cols and 'STATE' in cols
has_sa2 = 'SA2_CODE_2021' in cols and 'SA2_NAME_2021' in cols

if has_locality and has_sa2:
    locality_key = ['POSTCODE', 'STATE', 'LOCALITY_NAME']
    sa2_key = ['SA2_CODE_2021', 'SA2_NAME_2021']

    # Locality -> how many distinct SA2s?
    locality_to_sa2 = (
        merged.groupby(locality_key)[sa2_key]
        .nunique()
        .rename(columns={'SA2_CODE_2021': 'distinct_sa2_codes', 'SA2_NAME_2021': 'distinct_sa2_names'})
        .reset_index()
    )
    locality_multi = locality_to_sa2.query('distinct_sa2_codes > 1 or distinct_sa2_names > 1')

    # SA2 -> how many distinct localities?
    sa2_to_locality = (
        merged.groupby(sa2_key)[['POSTCODE', 'STATE', 'LOCALITY_NAME']]
        .nunique()
        .rename(columns={'POSTCODE': 'distinct_postcodes', 'STATE': 'distinct_states', 'LOCALITY_NAME': 'distinct_localities'})
        .reset_index()
    )
    sa2_multi = sa2_to_locality.query('distinct_postcodes > 1 or distinct_states > 1 or distinct_localities > 1')

    print('Localities mapping to multiple SA2s:', len(locality_multi))
    display(locality_multi.head(20))

    print('SA2s mapping to multiple localities/postcodes:', len(sa2_multi))
    display(sa2_multi.head(20))
else:
    print('Expected columns not present; check merge output.')


In [ ]:
# Outlier prep: create SA2-level frame and numeric income
assert INCOME_COL is not None, "Income column not detected. Check column selection in cell 0."

sa2_income_2024 = (
    merged[['SA2_CODE_2021', 'SA2_NAME_2021', INCOME_COL]]
    .drop_duplicates()
    .copy()
)

sa2_income_2024['income_numeric'] = pd.to_numeric(sa2_income_2024[INCOME_COL], errors='coerce')

coverage = {
    'sa2_rows': len(sa2_income_2024),
    'income_non_null': int(sa2_income_2024['income_numeric'].notna().sum()),
    'income_null': int(sa2_income_2024['income_numeric'].isna().sum()),
}
coverage


In [ ]:
# Distribution stats for SA2 income (2024)
stats = sa2_income_2024['income_numeric'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
stats


In [ ]:
# IQR-based outliers
q1 = sa2_income_2024['income_numeric'].quantile(0.25)
q3 = sa2_income_2024['income_numeric'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

sa2_income_2024['outlier_iqr_low'] = sa2_income_2024['income_numeric'] < lower
sa2_income_2024['outlier_iqr_high'] = sa2_income_2024['income_numeric'] > upper

summary_iqr = {
    'Q1': q1,
    'Q3': q3,
    'IQR': iqr,
    'LowerFence': lower,
    'UpperFence': upper,
    'num_low': int(sa2_income_2024['outlier_iqr_low'].sum()),
    'num_high': int(sa2_income_2024['outlier_iqr_high'].sum()),
}

extreme_low = sa2_income_2024.loc[sa2_income_2024['outlier_iqr_low']].sort_values('income_numeric').head(20)
extreme_high = sa2_income_2024.loc[sa2_income_2024['outlier_iqr_high']].sort_values('income_numeric', ascending=False).head(20)

summary_iqr, extreme_low[['SA2_CODE_2021','SA2_NAME_2021','income_numeric']].head(10), extreme_high[['SA2_CODE_2021','SA2_NAME_2021','income_numeric']].head(10)


In [ ]:
# Z-score outliers and mapping flags back to localities
mean_val = sa2_income_2024['income_numeric'].mean()
std_val = sa2_income_2024['income_numeric'].std(ddof=0)
sa2_income_2024['z_score'] = (sa2_income_2024['income_numeric'] - mean_val) / std_val

# 3-sigma threshold 
z_thresh = 3.0
sa2_income_2024['outlier_z'] = sa2_income_2024['z_score'].abs() > z_thresh

z_summary = {
    'mean': mean_val,
    'std': std_val,
    'z_thresh': z_thresh,
    'num_z_outliers': int(sa2_income_2024['outlier_z'].sum()),
}

# Map flags back to localities
flags = sa2_income_2024[['SA2_CODE_2021','outlier_iqr_low','outlier_iqr_high','outlier_z']]
merged_with_flags = merged.merge(flags, on='SA2_CODE_2021', how='left')

# Show some flagged localities
flagged_localities = merged_with_flags[
    merged_with_flags[['outlier_iqr_low','outlier_iqr_high','outlier_z']].any(axis=1)
][['POSTCODE','STATE','LOCALITY_NAME','SA2_CODE_2021','SA2_NAME_2021',INCOME_COL,'outlier_iqr_low','outlier_iqr_high','outlier_z']]

z_summary, flagged_localities.head(20)


In [ ]:
# Create 5 income bins (quintiles) and map to localities
# Quintiles give roughly equal-sized groups: Very Low → Very High
values = sa2_income_2024['income_numeric']
qres = pd.qcut(values, q=5, duplicates='drop')
num_bins = qres.cat.categories.size
labels_all = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
labels = labels_all[:num_bins]

sa2_income_2024['income_quintile'] = pd.qcut(values, q=num_bins, labels=labels, duplicates='drop')

# Optional: also store a numeric rank (1..num_bins)
sa2_income_2024['income_quintile_rank'] = sa2_income_2024['income_quintile'].cat.codes + 1

# Map bins back to localities
merged_with_bins = merged.merge(
    sa2_income_2024[['SA2_CODE_2021','income_quintile','income_quintile_rank']],
    on='SA2_CODE_2021',
    how='left'
)

# Show counts per bin and a preview
bin_counts = sa2_income_2024['income_quintile'].value_counts(dropna=False).sort_index()
bin_counts, merged_with_bins[['POSTCODE','STATE','LOCALITY_NAME','SA2_NAME_2021','income_quintile','income_quintile_rank']].head(20)
